In [ ]:
import torch
import numpy as np
import pandas as pd
import os, glob, json, csv,unicodedata, re
import matplotlib.pyplot as plt
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)
from sklearn.model_selection import StratifiedKFold

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
file_paths = [
    '/content/drive/MyDrive/Data/A1_mynediad-A2_sylfaen-de-learnwelsh.json',
    '/content/drive/MyDrive/Data/B1_canolradd_de-learnwelsh.json',
    '/content/drive/MyDrive/Data/B2_uwch-1_de-learnwelsh.json',
    '/content/drive/MyDrive/Data/B2_Uwch-2_de-learnwelsh.json',
    '/content/drive/MyDrive/Data/B2_Uwch-3_de-learnwelsh.json'
]

In [ ]:
# Read and combine all data
df_allData_list = []

for path in file_paths:
    df_allData = pd.read_json(path)
    df_allData = df_allData.dropna().drop_duplicates(subset="text")
    df_allData_list.append(df_allData)

# Combine all DataFrames into one
df_allData = pd.concat(df_allData_list, ignore_index=True)

# Reset index for cleanliness
df_allData = df_allData.reset_index(drop=True)

print(f"Combined dataset shape: {df_allData.shape}")
df_allData.head()

In [ ]:
df_allData["cefr_level"].value_counts()

In [ ]:
# HuggingFace Dataset
ds_merged = Dataset.from_pandas(df_allData)

In [ ]:
ds_merged

In [ ]:
CEFR_LEVELS = ["A1", "A2","B1","B2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}
labels = np.array([label2id[l] for l in ds_merged["cefr_level"]])

In [ ]:
model_name = "EuroBERT/EuroBERT-210m"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

In [ ]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [ ]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [ ]:
# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

In [ ]:
# Text Cleaning
def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Normalize accents and remove control characters
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\u00A0", " ")  # Replace NBSP with space
    text = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]", "", text)
    return text

# Normalize for single-line export (Excel/CSV safe)
def to_one_line(text):

    text = clean_text(text)
    text = re.sub(r"[\r\n]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
# W&B auto-login and any remote telemetry
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

best_f1 = 0.0
best_trainer = None
best_tokenizer = None

all_cms = []
all_y_true, all_y_pred = [], []
all_misclassified = []

RUN_ROOT = "./eurobert_cefr_welsh_allData"
os.makedirs(RUN_ROOT, exist_ok=True)

id2label = dict(enumerate(CEFR_LEVELS))
labels_order = list(range(len(CEFR_LEVELS)))

In [ ]:
def plot_cm(cm, labels, title, outfile, normalize=False):
    arr = cm.astype(float)
    if normalize:
        arr = arr / arr.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(6,5))
    im = ax.imshow(arr, aspect='auto')
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(len(labels)), yticks=np.arange(len(labels)),
        xticklabels=labels, yticklabels=labels,
        xlabel="Predicted", ylabel="True", title=title
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    thresh = arr.max() / 2.0 if arr.size else 0
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            txt = f"{arr[i,j]:.2f}" if normalize else f"{int(cm[i,j])}"
            ax.text(j, i, txt, ha="center", va="center",
                    color="white" if arr[i,j] > thresh else "black")
    fig.tight_layout(); fig.savefig(outfile, dpi=200); plt.close(fig)

In [ ]:
for fold, (train_idx, val_idx) in enumerate(skf.split(ds_merged, labels), start=1):
    print(f"\n Running Fold {fold}...")

    ds_train = ds_merged.select(train_idx)
    ds_val = ds_merged.select(val_idx)

    tok_train = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)
    tok_val = ds_val.map(preprocess, batched=True, remove_columns=ds_val.column_names)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(CEFR_LEVELS), trust_remote_code=True
    )

    args = TrainingArguments(
        output_dir=f"{RUN_ROOT}/fold_{fold}",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=3,
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_weighted_f1",
        greater_is_better=True,
        seed=42,
        learning_rate=3.6e-5,
        warmup_ratio=0.1,
        gradient_accumulation_steps=16,
        optim="adamw_torch_fused",
        lr_scheduler_type="linear",
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        bf16=False
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    # Track best trainer
    if metrics["eval_weighted_f1"] > best_f1:
        best_f1 = metrics["eval_weighted_f1"]
        best_trainer = trainer
        best_tokenizer = tokenizer

    # Predictions for CM + misclassified
    pred = trainer.predict(tok_val)
    logits = pred.predictions if isinstance(pred.predictions, np.ndarray) else pred.predictions[0]
    y_true = pred.label_ids
    if logits.ndim > 1:
        # confidence via softmax max (stable, simple)
        s = np.exp(logits - logits.max(axis=1, keepdims=True))
        probs = s / s.sum(axis=1, keepdims=True)
        y_pred = probs.argmax(axis=1)
        y_conf = probs.max(axis=1)
    else:
        y_pred = (logits > 0.5).astype(int)
        y_conf = np.maximum(logits, 1 - logits)

    # Per-fold CM
    cm = confusion_matrix(y_true, y_pred, labels=labels_order)
    all_cms.append(cm)
    all_y_true.append(y_true)
    all_y_pred.append(y_pred)

    # Per-fold misclassified (tidy text for Excel)
    fold_texts = ds_val["text"] if "text" in ds_val.column_names else [""] * len(y_true)
    fold_texts = [to_one_line(t) for t in fold_texts]
    fold_sources = ds_val["source_name"] if "source_name" in ds_val.column_names else ["unknown"] * len(y_true)

    df_fold = pd.DataFrame({
        "fold": fold,
        "source_name": fold_sources,
        "text": fold_texts,
        "true_label": [id2label[i] for i in y_true],
        "pred_label": [id2label[i] for i in y_pred],
        "pred_confidence": y_conf,
    })
    all_misclassified.append(df_fold[df_fold["true_label"] != df_fold["pred_label"]])

    # Store fold metrics
    row = {
        "Fold": fold,
        "All CEFR Levels Precision": metrics.get("eval_weighted_precision", 0.0),
        "All CEFR Levels Recall": metrics.get("eval_weighted_recall", 0.0),
        "All CEFR Levels F1": metrics.get("eval_weighted_f1", 0.0),
    }
    for level in CEFR_LEVELS:
        row[f"{level} Precision"] = metrics.get(f"eval_{level}_precision", 0.0)
        row[f"{level} Recall"] = metrics.get(f"eval_{level}_recall", 0.0)
        row[f"{level} F1"] = metrics.get(f"eval_{level}_f1", 0.0)
    all_results.append(row)

In [ ]:
# Overall aggregation (preds + CM)
all_y_true = np.concatenate(all_y_true)
all_y_pred = np.concatenate(all_y_pred)

lab_names = [id2label[i] for i in labels_order]

overall_cm = confusion_matrix(all_y_true, all_y_pred, labels=labels_order)
pd.DataFrame(overall_cm, index=CEFR_LEVELS, columns=CEFR_LEVELS).to_csv(
    f"{RUN_ROOT}/confusion_matrix_overall.csv", index=True, encoding="utf-8-sig", lineterminator="\n"
)
plot_cm(
    overall_cm, CEFR_LEVELS, "Overall Confusion Matrix (Counts)",
    f"{RUN_ROOT}/cm_overall_counts.png", normalize=False
)
plot_cm(
    overall_cm, CEFR_LEVELS, "Overall Confusion Matrix (Row-Normalized)",
    f"{RUN_ROOT}/cm_overall_rownorm.png", normalize=True
)

# Overall misclassifications
mis_all = (
    pd.concat(all_misclassified, ignore_index=True)
    if len(all_misclassified) else
    pd.DataFrame(columns=["fold","source_name","text","true_label","pred_label","pred_confidence"])
)
mis_all["is_mis"] = mis_all["true_label"] != mis_all["pred_label"]
mis_all = mis_all[mis_all["is_mis"]].drop(columns=["is_mis"])

# Confidence bins (quantiles across all misclassified)
N_BINS = 5
if len(mis_all):
    q = pd.qcut(mis_all["pred_confidence"], q=N_BINS, duplicates="drop")
    mis_all["conf_bin"] = q.cat.codes
    mis_all["conf_interval"] = q.astype(str)         # readable interval
else:
    mis_all["conf_bin"] = pd.Series(dtype="int64")
    mis_all["conf_interval"] = pd.Series(dtype="string")

# Save misclassified overall (sorted by confidence)
mis_all.sort_values("pred_confidence", ascending=False).to_csv(
    f"{RUN_ROOT}/misclassified_overall.csv", index=False, encoding="utf-8-sig", lineterminator="\n"
)

# Bin summary
if len(mis_all):
    bin_summary = (
        mis_all.groupby("conf_bin", as_index=False)
               .agg(
                   n=("pred_confidence","size"),
                   min_conf=("pred_confidence","min"),
                   max_conf=("pred_confidence","max"),
                   mean_conf=("pred_confidence","mean")
               )
               .sort_values("conf_bin")
    )
    # attach interval strings (first match per bin)
    intervals = mis_all.groupby("conf_bin")["conf_interval"].first().reset_index()
    bin_summary = bin_summary.merge(intervals, on="conf_bin", how="left")

    bin_summary.to_csv(
        f"{RUN_ROOT}/misclassified_overall_bin_summary.csv",
        index=False, encoding="utf-8-sig", lineterminator="\n"
    )

print(f"\nDone. Files in {RUN_ROOT}:")
print("confusion_matrix_overall.csv")
print(" - cm_overall_counts.png")
print(" - cm_overall_rownorm.png")
print(" - misclassified_overall.csv")
print(" - misclassified_overall_bin_summary.csv" if len(mis_all) else " - (no misclassifications; no bin summary)")
print(f"Best eval_weighted_f1 observed: {best_f1:.4f}")

In [ ]:
# Save best-performing model from all folds
final_path = "./eurobert_cefr_welsh_allData/best_model"
best_trainer.save_model(final_path)
best_tokenizer.save_pretrained(final_path)
best_trainer.state.save_to_json(os.path.join(final_path, "trainer_state.json"))

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(all_results)

# Compute average row
average_row = df.drop(columns=["Fold"]).mean(numeric_only=True)
average_row["Fold"] = "Average"
df = pd.concat([df, pd.DataFrame([average_row])], ignore_index=True)

# Restructure columns
columns = [("Fold", "")] + [
    ("All CEFR Levels", "Precision"), ("All CEFR Levels", "Recall"), ("All CEFR Levels", "F1"),
    ("A1", "Precision"), ("A1", "Recall"), ("A1", "F1"),
    ("A2", "Precision"), ("A2", "Recall"), ("A2", "F1"),
    ("B1", "Precision"), ("B1", "Recall"), ("B1", "F1"),
    ("B2", "Precision"), ("B2", "Recall"), ("B2", "F1"),
]


df = df[[col[0] if col[1] == "" else f"{col[0]} {col[1]}" for col in columns]]
df.columns = pd.MultiIndex.from_tuples(columns)

In [ ]:
df